# 08 — Document Classifier: Auto-Sort Incoming Insurance Documents

**Time**: ~15 minutes · **Custom classifier** · **Requires labeled training data**

---

## Insurance Scenario

An insurance company receives a **mixed batch of documents** in a single submission:
- Claim forms
- Invoices
- ID documents
- Medical reports
- Police reports

Before extraction, you need to **classify each document** to route it to the correct processing pipeline. A custom classifier identifies document types (classes) automatically.

---

## Prerequisites

1. Complete [00-setup-and-basics.ipynb](00-setup-and-basics.ipynb)
2. Training data organized in folders (one per document class) in Azure Blob Storage

### Training Data Structure

```
training-data/
├── claim-forms/        (5+ samples)
│   ├── claim1.pdf
│   └── claim2.pdf
├── invoices/           (5+ samples)
│   ├── invoice1.pdf
│   └── invoice2.pdf
├── id-documents/       (5+ samples)
│   ├── license1.png
│   └── passport1.jpg
└── medical-reports/    (5+ samples)
    ├── report1.pdf
    └── report2.pdf
```

In [ ]:
import os
import uuid
from dotenv import load_dotenv
from azure.core.credentials import AzureKeyCredential
from azure.ai.documentintelligence import (
    DocumentIntelligenceClient,
    DocumentIntelligenceAdministrationClient,
)
from azure.ai.documentintelligence.models import (
    BuildDocumentClassifierRequest,
    ClassifierDocumentTypeDetails,
    AzureBlobContentSource,
    ClassifyDocumentRequest,
    AnalyzeResult,
)

from azure.identity import DefaultAzureCredential

load_dotenv(dotenv_path=os.path.join("..", ".env"))

endpoint = os.environ["DOCUMENTINTELLIGENCE_ENDPOINT"]
_api_key = os.environ.get("DOCUMENTINTELLIGENCE_API_KEY", "")
credential = AzureKeyCredential(_api_key) if _api_key else DefaultAzureCredential()

admin_client = DocumentIntelligenceAdministrationClient(
    endpoint=endpoint, credential=credential
)
client = DocumentIntelligenceClient(
    endpoint=endpoint, credential=credential
)

print("✅ Clients ready")

## Step 1: Build a Custom Classifier

A classifier needs at least **5 samples per document class**. You can use [Document Intelligence Studio](https://documentintelligence.ai.azure.com/studio) for a visual labeling experience.

In [ ]:
# ⚠️ Uncomment when you have training data organized in Azure Blob Storage
#
# container_sas_url = os.environ["DOCUMENTINTELLIGENCE_STORAGE_CONTAINER_SAS_URL"]
# classifier_id = f"insurance-doc-classifier-{uuid.uuid4().hex[:8]}"
#
# # Define document classes — each maps to a subfolder in blob storage
# poller = admin_client.begin_build_classifier(
#     BuildDocumentClassifierRequest(
#         classifier_id=classifier_id,
#         description="Classifies incoming insurance documents",
#         doc_types={
#             "claim-form": ClassifierDocumentTypeDetails(
#                 azure_blob_source=AzureBlobContentSource(
#                     container_url=container_sas_url,
#                     prefix="claim-forms/"
#                 )
#             ),
#             "invoice": ClassifierDocumentTypeDetails(
#                 azure_blob_source=AzureBlobContentSource(
#                     container_url=container_sas_url,
#                     prefix="invoices/"
#                 )
#             ),
#             "id-document": ClassifierDocumentTypeDetails(
#                 azure_blob_source=AzureBlobContentSource(
#                     container_url=container_sas_url,
#                     prefix="id-documents/"
#                 )
#             ),
#             "medical-report": ClassifierDocumentTypeDetails(
#                 azure_blob_source=AzureBlobContentSource(
#                     container_url=container_sas_url,
#                     prefix="medical-reports/"
#                 )
#             ),
#         },
#     )
# )
# classifier = poller.result()
#
# print(f"✅ Classifier trained: {classifier.classifier_id}")
# print(f"   Classes: {list(classifier.doc_types.keys())}")

print("Uncomment the code above when you have training data ready.")
print("\nTo prepare training data:")
print("1. Organize documents into folders by type (5+ samples each)")
print("2. Upload to Azure Blob Storage")
print("3. Generate a container SAS URL")

## Step 2: Classify a Document

Send a document to the classifier to determine its type.

In [ ]:
# ⚠️ Uncomment after training a classifier (Step 1)
#
# classifier_id = "insurance-doc-classifier-xxxxxxxx"  # Your classifier ID
#
# # Classify a URL-based document
# document_url = "https://raw.githubusercontent.com/Azure-Samples/cognitive-services-REST-api-samples/master/curl/form-recognizer/sample-invoice.pdf"
#
# poller = client.begin_classify_document(
#     classifier_id,
#     ClassifyDocumentRequest(url_source=document_url)
# )
# result: AnalyzeResult = poller.result()
#
# print("Classification Results:")
# print("=" * 50)
# if result.documents:
#     for doc in result.documents:
#         print(f"  Document type: {doc.doc_type}")
#         print(f"  Confidence:    {doc.confidence:.2%}")
#         if doc.bounding_regions:
#             pages = [str(r.page_number) for r in doc.bounding_regions]
#             print(f"  Pages:         {', '.join(pages)}")
#         print()

print("Uncomment the code above after training a classifier.")

## Step 3: Build a Classification-to-Extraction Pipeline

**This is the real power for insurance**: Classify first, then route to the right extraction model.

In [ ]:
# Pattern: Classify → Route → Extract
#
# MODEL_ROUTING = {
#     "claim-form":     "insurance-claim-form-model",   # Custom model
#     "invoice":        "prebuilt-invoice",              # Prebuilt
#     "id-document":    "prebuilt-idDocument",           # Prebuilt
#     "medical-report": "prebuilt-layout",               # Layout for unstructured
# }
#
# def classify_and_extract(document_url, classifier_id):
#     """Classify a document and route to the appropriate extraction model."""
#     # Step 1: Classify
#     classify_poller = client.begin_classify_document(
#         classifier_id,
#         ClassifyDocumentRequest(url_source=document_url)
#     )
#     classify_result = classify_poller.result()
#     
#     results = []
#     if classify_result.documents:
#         for doc in classify_result.documents:
#             doc_type = doc.doc_type
#             model_id = MODEL_ROUTING.get(doc_type, "prebuilt-layout")
#             
#             # Step 2: Extract with the appropriate model
#             extract_poller = client.begin_analyze_document(
#                 model_id,
#                 AnalyzeDocumentRequest(url_source=document_url)
#             )
#             extract_result = extract_poller.result()
#             
#             results.append({
#                 "document_type": doc_type,
#                 "classification_confidence": doc.confidence,
#                 "extraction_model": model_id,
#                 "extraction_result": extract_result,
#             })
#     return results

print("The classify → route → extract pattern above shows how to build")
print("an end-to-end document processing pipeline for insurance.")
print("")
print("This is especially useful for:")
print("  • Mail room automation (mixed incoming documents)")
print("  • Claims intake (policyholder submits multiple docs)")
print("  • Document digitization projects")

---

## Key Takeaways

| Feature | Value for Insurance |
|---|---|
| **Multi-class classification** | Sort incoming mail into claim forms, invoices, IDs, reports |
| **Per-page classification** | Handle multi-document PDFs (e.g., claim package with multiple attachments) |
| **Pipeline routing** | Auto-route to the right extraction model based on document type |
| **Custom classes** | Define classes for your specific document types |

## Next Steps

| Notebook | What You'll Learn |
|---|---|
| [09-addon-capabilities.ipynb](09-addon-capabilities.ipynb) | Barcodes, query fields, and high-res mode |
| [10-human-in-the-loop.ipynb](10-human-in-the-loop.ipynb) | Add human review to your pipeline |